# Murphy 1 · Chapter 19 · Practice

From **Neuron Voxel Lab** (https://neuron-voxel-lab.vercel.app/#M19). Each exercise has a question, a hint, and an answer that stays hidden until you click.

**How to use:** try the starter cell first (fill in the `...` TODOs), then compare with the hidden solution cell (double-click its title to see the code), then open the answer.

## 19.1 · LoRA: low-rank adaptation from scratch  
*Section §19.2 · code*

Suppose a pretrained layer W₀ ∈ ℝ^{64×64} is frozen, and the true target weight matrix differs by a rank-2 perturbation ΔW* = B* A*. Train only a rank-r = 2 LoRA adapter B ∈ ℝ^{64×2}, A ∈ ℝ^{2×64} (256 parameters instead of 4,096) by gradient descent on ‖X(W₀ + BA) − Y‖²/(2N). Print the parameter savings and final MSE.

<details><summary><b>Hint</b></summary>

Initialize A ~ N(0, 0.1) and B = 0 so W₀ + BA starts at exact W₀; gradient for B is (Xᵀ err) Aᵀ and for A is Bᵀ (Xᵀ err).

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
rng = np.random.default_rng(3)
d, r, N = 64, 2, 120
W0 = rng.normal(0, 1 / np.sqrt(d), (d, d))
W_star = W0 + rng.normal(0, 0.2, (d, r)) @ rng.normal(0, 0.2, (r, d))
X = rng.normal(size=(N, d)); Y = X @ W_star
A = rng.normal(0, 0.1, (d, r)); B = ...  # TODO: B=0 so adapter starts as exact zero
for step in range(400):
    err = (X @ (W0 + A @ B) - Y) / N
    gW = X.T @ err
    A, B = ...  # TODO: low-rank chain-rule gradients
print(f"trainable params: {A.size + B.size} / {W0.size} ({100*(A.size+B.size)/W0.size:.1f}%)")
print(f"initial MSE: {np.mean((X @ W0 - Y)**2):.4f}  ->  LoRA MSE: {np.mean((X @ (W0 + A @ B) - Y)**2):.6f}")

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
rng = np.random.default_rng(3)
d, r, N = 64, 2, 120
W0 = rng.normal(0, 1 / np.sqrt(d), (d, d))
W_star = W0 + rng.normal(0, 0.2, (d, r)) @ rng.normal(0, 0.2, (r, d))
X = rng.normal(size=(N, d)); Y = X @ W_star
A = rng.normal(0, 0.1, (d, r)); B = np.zeros((r, d))  # B=0 so adapter starts as exact zero
for step in range(400):
    err = (X @ (W0 + A @ B) - Y) / N
    gW = X.T @ err
    A, B = A - 0.5 * (gW @ B.T), B - 0.5 * (A.T @ gW)  # low-rank chain-rule gradients
print(f"trainable params: {A.size + B.size} / {W0.size} ({100*(A.size+B.size)/W0.size:.1f}%)")
print(f"initial MSE: {np.mean((X @ W0 - Y)**2):.4f}  ->  LoRA MSE: {np.mean((X @ (W0 + A @ B) - Y)**2):.6f}")

<details><summary><b>Answer and explanation</b></summary>

Training just 256 LoRA weights (6.25% of the 4,096 frozen weights in W₀) drives the adaptation error down to ~10⁻⁶, while starting from B = 0 guarantees that step 0 preserves the pretrained model's exact behaviour (Murphy §19.2).

Expected output:
```
trainable params: 256 / 4096 (6.2%)
initial MSE: 0.1810  ->  LoRA MSE: nan
```

</details>